In [1]:
import pandas as pd 
import os 
import logging 
import traceback
from basicprocess import create_folder, outputlog
from TDXdataframe import read_businfo_xml

In [2]:
def get_taipeibusreport(odspath):

    logging.info("開始讀取台北市營運月報")
    logging.info(f"台北市公車營運月報的原始檔案為{odspath}")

    sheet_names = pd.ExcelFile(odspath).sheet_names
    dfs = []
    for sheet in sheet_names:
        odsdf = pd.read_excel(odspath, sheet_name=sheet, engine='odf')
        odsdf['Time'] = sheet
        dfs.append(odsdf)

    df = pd.concat(dfs, ignore_index=True)

    logging.info("台北市營運月報整併完成")
    

    df['年'] = df['Time'].str[:3].astype(int) + 1911
    df['月'] = df['Time'].str[3:].astype(int)
    # df['資料時間'] = pd.to_datetime(df['年'].astype(str) + '-' + df['月'].astype(str)).dt.to_period('M')
    df['資料時間'] = pd.to_datetime(
        df['年'].astype(str) + '-' + df['月'].astype(str).str.zfill(2),
        format='%Y-%m',
        errors='raise').dt.to_period('M')

    cols = ['資料時間'] + [c for c in df.columns if c not in ['年', '月', 'Time', '資料時間']]
    df = df.reindex(columns=cols)

    logging.info("輸出指定格式")

    rename_dist = {'資料時間':'Month',
                '客運業者':'OperatorName', 
                '路線代碼':'RouteID', 
                '路線別':'RouteName', 
                '總班次':'Shifts', 
                '總載客人次':'Population',
                '總行駛里程':'Miles',
                '延人公里':'PaxKm', 
                '總營收': 'Revenue'}
    df = df[list(rename_dist)]
    df = df.rename(columns = rename_dist)

    return df

def check_if_morethanone(df, checklists, timecolumn, warningfolder, dataname = '資料'):
    """
    檢查每個 timecolumn 內，checklists 是否有重複值

    Parameters
    ----------
    df : pandas.DataFrame
    checklists : list
        需要檢查是否重複的欄位
    timecolumn : str
        時間欄位（例如 Month）

    Returns
    -------
    duplicated_df : pandas.DataFrame
        含有重複資料的 dataframe（只保留重複者）
    summary : pandas.DataFrame
        每個月份重複筆數的摘要
    """

    # 找出在「同一個月 + checklists」下重複的資料
    mask = df.duplicated(subset=[timecolumn] + checklists, keep=False)
    duplicated_df = df[mask].sort_values([timecolumn] + checklists)

    if len(duplicated_df) > 0:
        logging.warning(f"{dataname} 有重複的資料")

        # 每月重複筆數摘要
        summary = (
            duplicated_df
            .groupby(timecolumn)
            .size()
            .reset_index(name='DuplicatedRows')
        )

        outputfile = os.path.join(warningfolder, f"{dataname}重複資料.xlsx")

        with pd.ExcelWriter(outputfile, engine='xlsxwriter') as writer:
            duplicated_df.to_excel(writer, index=True, sheet_name='有重複的資料')
            summary.to_excel(writer, index=True, sheet_name='每月重複筆數')

        logging.info(f"{dataname}路線營運月報，路徑：{outputfile}")

        return False     

    else:
        logging.info(f"{dataname}在{checklists}的組合底下沒有重複資料")
        return True



In [3]:
taipeidf = pd.read_excel(r"D:\B-Project\2025\6800\Technical\12票證資料\TicketAnalysis\03_處理後資料\03_公車營運月報\臺北市公車營運月報.xlsx")

In [4]:

taipeidf.drop(columns='OperatorName').groupby(['Month','RouteID', 'RouteName']).agg({'Shifts':'sum',
                                                                                     'Population':'sum',
                                                                                     'Miles':'sum',
                                                                                     'PaxKm':'sum',
                                                                                     'Revenue':'sum'}).reset_index()

,Month,RouteID,RouteName,Shifts,Population,Miles,PaxKm,Revenue
0,2024-01,1,小1路,496,4736,5134,39214,111661
1,2024-01,2,小2路,352,4200,5051,48216,99025
2,2024-01,2,紅2接駁,1882,61585,30206,790751,1451987
3,2024-01,3,小3路,1600,21396,21120,225942,504455
4,2024-01,3,紅3區間車,1758,50031,26370,300186,1179580
...,...,...,...,...,...,...,...,...
7599,2025-06,紅25,紅25,876,35072,10775,345108,891549
7600,2025-06,紅33,紅33,342,3609,2924,24686,91743
7601,2025-06,紅50,紅50,1632,20103,13709,135092,511022
7602,2025-06,藍10,藍10,3241,107115,34030,899766,2722910


In [ ]:
'''Setup & Main Execution'''
# 00_Setup 所有全域函數
logfile = os.path.abspath(os.path.join(os.getcwd(), '..', 'Log', '04_營運月報整理.log'))
if os.path.exists(logfile):
    os.remove(logfile)
    
logging.basicConfig(
    filename=logfile,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

def main():
    logging.info('Start Processing...')

    outputfolder = create_folder(os.path.join(os.getcwd(), '..', '03_處理後資料'))
    monthlyreport_organized_folder = create_folder(os.path.join(outputfolder, '03_公車營運月報'))
    
    # 處理台北公車路線資料
    taipeidf = get_taipeibusreport(r"D:\OneDrive - 鼎漢國際工程顧問股份有限公司\部門空間管理者8\B-6812-修訂臺北都會區整體路網\Technical\18票證月報資料\04臺北\01公運處\附2-項目(二)臺北市營運資料(市區公車)v1.ods")
    taipeidf.to_excel(os.path.join(monthlyreport_organized_folder, '臺北市公車營運月報.xlsx'), index = False)
    logging.info('輸出臺北市公車營運月報整理結果')
    temp = check_if_morethanone(df = taipeidf, 
                                checklists = ['RouteID', 'RouteName' ], 
                                timecolumn = 'Month', 
                                dataname = '臺北市公車營運月報', 
                                warningfolder=create_folder(os.path.join(monthlyreport_organized_folder, 'error')))
    if temp == False:
        logging.info("臺北市公車因為有多個OpeartionName經營同一條路線")
        taipeidf = taipeidf.drop(columns='OperatorName').groupby(['Month','RouteID', 'RouteName']).agg({'Shifts':'sum',
                                                                                                        'Population':'sum',
                                                                                                        'Miles':'sum',
                                                                                                        'PaxKm':'sum',
                                                                                                        'Revenue':'sum'}).reset_index()

        taipeidf.to_excel(os.path.join(monthlyreport_organized_folder, '臺北市公車營運月報.xlsx'), index = False)
        logging.info('重新輸出臺北市公車營運月報整理結果')

    del temp


    logging.info('Finished Processing.')


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        logging.error("main() 執行失敗：%s", e)  
        logging.error("Traceback:\n%s", traceback.format_exc())
    
    outputlog(logfile=logfile)